# CellLineFinder — Scoring Formula: Implementation & Real-Data Test

This notebook implements and stress-tests the confidence-score formula
from `CellLineSelector_技术方案与需求分析.docx`:

```
Final Score = 0.40 x RNA expression
            + 0.25 x Protein evidence
            + 0.15 x Data consistency
            + 0.10 x Tissue relevance
            + 0.10 x Data completeness
            - Exclusion-gene penalty
```

The formula in the spec doc is a design sketch — it doesn't define exactly
*how* each term should be computed, or what should happen when a term's
underlying data is missing. This notebook surfaces those gaps using real
data, and documents the decisions made in `scoring.py` to resolve them.


In [8]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from data_loader import CellLineDataLoader
from scoring import score_gene_profile, DEFAULT_WEIGHTS
import pandas as pd

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 140)

DATA_DIR = Path.cwd().parent / "data"
loader = CellLineDataLoader(DATA_DIR)
print("Loader ready:", loader.data_dir.resolve())

Loader ready: /Users/liyannan/Desktop/Data Science MSc Graduation Project/data


## 1. The gap the spec doesn't cover: protein data coverage

Before scoring anything, it's worth checking how often RNA *and* protein
data are actually both available for the same cell line — because the
formula assumes both exist for every row.

In [9]:
profile = loader.build_gene_profile("EGFR")

has_rna = profile["rna_expression"].notna()
has_protein = profile["protein_expression"].notna()

print(f"Total candidates: {len(profile)}")
print(f"RNA only:        {(has_rna & ~has_protein).sum()}  ({(has_rna & ~has_protein).mean():.0%})")
print(f"Protein only:    {(~has_rna & has_protein).sum()}  ({(~has_rna & has_protein).mean():.0%})")
print(f"Both:            {(has_rna & has_protein).sum()}  ({(has_rna & has_protein).mean():.0%})")

corr = profile.loc[has_rna & has_protein, ["rna_expression", "protein_expression"]].corr().iloc[0,1]
print(f"\nPearson correlation (RNA vs protein), cell lines with both: {corr:.3f}")

[data_loader] Warning: 152885 of 1066869 mutation records could not be mapped to an ACH-ID (ProfileID missing from file 8). These rows are dropped.
Total candidates: 1434
RNA only:        1058  (74%)
Protein only:    6  (0%)
Both:            370  (26%)

Pearson correlation (RNA vs protein), cell lines with both: 0.860


**Finding:** only ~26% of EGFR candidates have both RNA and protein
measurements — proteomics (file 4) covers far fewer cell lines than
RNA-seq (file 2). The 0.86 correlation where both exist confirms "protein
evidence" and "data consistency" are real signals worth keeping in the
formula — but scoring the missing 74% as a flat 0 on those terms would
penalise cell lines for not being *measured*, not for actually having low
expression. That's a confound, not a finding.

**Resolution implemented in `scoring.py`:** for rows missing protein data,
the `protein` and `consistency` weights are redistributed onto `rna` and
`tissue`, proportionally — so a protein-less cell line is judged on the
evidence that exists, not punished for a gap in the proteomics study's
coverage.

## 2. Running the scorer (default weights, as specified)

In [10]:
GENE = "EGFR"
profile = loader.build_gene_profile(GENE, exclude_genes=["KRAS"])
scored = score_gene_profile(profile)

cols = ["DepMap_ID", "cell_line_name", "lineage", "rna_expression",
        "protein_expression", "final_score", "excluded"]
scored[cols].head(15)

,DepMap_ID,cell_line_name,lineage,rna_expression,protein_expression,final_score,excluded
0,ACH-000741,U-BLC1,urinary_tract,10.076148,5.154394,0.947452,False
1,ACH-000012,HCC827,lung,9.673645,2.724113,0.825375,False
2,ACH-000777,KYSE-30,esophagus,8.359530,3.105255,0.819211,False
3,ACH-001411,UM-UC-5,urinary_tract,9.417620,NaN,0.810582,False
4,ACH-000637,KYSE-520,esophagus,9.286535,NaN,0.802989,False
5,ACH-000865,KYSE-450,esophagus,8.360803,2.363419,0.788122,False
6,ACH-000109,NCI-H3255,lung,8.925347,2.465903,0.786941,False
7,ACH-000693,KYSE-180,esophagus,7.200850,3.443853,0.784981,False
8,ACH-000546,HSC-4,upper_aerodigestive,7.692860,2.768229,0.783310,False
9,ACH-001523,HSC-1,skin,10.531869,NaN,0.782414,False


**Sanity check:** HCC827 and NCI-H3255 — both well-known EGFR-driven lung
cancer cell lines in the literature — land in the top 10. Several
oesophageal lines also cluster near the top, consistent with EGFR's known
role in oesophageal squamous carcinoma. This is a reasonable result, not
proof of correctness — worth a second sanity check against a gene with
well-documented cell line associations before relying on this for
real selection decisions.

## 3. Verifying the exclusion logic actually excludes

In [11]:
n_excluded = (scored["final_score"] == -1.0).sum()
print(f"Cell lines flagged for exclusion (KRAS mutation/fusion): {n_excluded} / {len(scored)}")
print(f"\nExcluded rows sink to the bottom but are kept (not dropped) for auditing:")
scored.loc[scored["final_score"] == -1.0, cols].head(5)

Cell lines flagged for exclusion (KRAS mutation/fusion): 237 / 1434

Excluded rows sink to the bottom but are kept (not dropped) for auditing:


,DepMap_ID,cell_line_name,lineage,rna_expression,protein_expression,final_score,excluded
1197,ACH-000599,PA-TU-8902,pancreas,5.056584,NaN,-1.0,True
1198,ACH-000651,SW 620,colorectal,0.097611,-0.762236,-1.0,True
1199,ACH-002467,RPE1-ss51,eye,4.498889,NaN,-1.0,True
1200,ACH-000941,HEC-1-B,uterus,4.554589,NaN,-1.0,True
1201,ACH-001849,ICC9,bile_duct,5.012569,NaN,-1.0,True


## 4. Two places the spec was ambiguous — both configurable

### 4a. "Tissue relevance" — relative to what?

The spec names this as a scoring term but never says what a cell line's
tissue is supposed to be *relevant to*. Two interpretations are
implemented:

In [12]:
# Interpretation A (default): relevance = how enriched this cell line's
# lineage is among the TOP RNA-expressing candidates for this gene.
# No target tissue needs to be specified.
scored_auto = score_gene_profile(profile)
print("Default (auto-detected tissue enrichment) - top 5:")
print(scored_auto[["cell_line_name", "lineage", "tissue_score", "final_score"]].head())

Default (auto-detected tissue enrichment) - top 5:
  cell_line_name        lineage  tissue_score  final_score
0         U-BLC1  urinary_tract      0.648649     0.947452
1         HCC827           lung      0.382488     0.825375
2        KYSE-30      esophagus      0.656250     0.819211
3        UM-UC-5  urinary_tract      0.648649     0.810582
4       KYSE-520      esophagus      0.656250     0.802989


In [13]:
# Interpretation B: explicit target tissue, as literally read from the spec
# ("tissue relevance" = does this match the tissue I care about).
scored_lung = score_gene_profile(profile, target_lineage="lung")
print("target_lineage='lung' - top 5:")
print(scored_lung[["cell_line_name", "lineage", "tissue_score", "final_score"]].head())

target_lineage='lung' - top 5:
  cell_line_name        lineage  tissue_score  final_score
0         HCC827           lung           1.0     0.887127
1         U-BLC1  urinary_tract           0.0     0.882587
2      NCI-H3255           lung           1.0     0.848693
3      NCI-H1568           lung           1.0     0.823742
4         HCC-95           lung           1.0     0.797325


**Recommendation for the team:** Interpretation B (explicit target
tissue) is probably the one users actually want — "find me a lung cancer
cell line with high EGFR" is a more natural query than "find me whatever
tissue happens to be enriched." Suggest adding `target_lineage` as a
user-facing input field rather than relying on auto-detection.

### 4b. Custom weight overrides

Useful for testing how sensitive the ranking is to the 0.40/0.25/0.15/0.10/0.10
split the spec proposed.

In [14]:
# Example: what if we trust RNA much more than protein (e.g. because
# protein coverage is so sparse)?
scored_rna_heavy = score_gene_profile(profile, weights={"rna": 0.6, "protein": 0.1})
print("rna=0.6, protein=0.1 - top 5:")
print(scored_rna_heavy[["cell_line_name", "final_score"]].head())

print("\nDefault weights for comparison - top 5:")
print(scored[["cell_line_name", "final_score"]].head())

rna=0.6, protein=0.1 - top 5:
  cell_line_name  final_score
0         U-BLC1     0.988797
1         HCC827     0.902823
2          HSC-1     0.873645
3        UM-UC-5     0.866167
4        KYSE-30     0.864842

Default weights for comparison - top 5:
  cell_line_name  final_score
0         U-BLC1     0.947452
1         HCC827     0.825375
2        KYSE-30     0.819211
3        UM-UC-5     0.810582
4       KYSE-520     0.802989


**Finding:** the top 2-3 candidates are stable across both weightings —
U-BLC1 and HCC827 stay on top either way. This is a reassuring sign that
the ranking isn't hypersensitive to the exact weight split, at least for
this gene. Worth re-checking with a gene that has weaker/noisier RNA-protein
agreement before concluding the formula is robust in general.

## 5. Summary for the team

**What works as specified:**
- RNA and protein min-max normalisation
- Exclusion-gene hard filtering (mutations + fusions)
- The overall 5-term weighted structure

**Where the spec needed a decision (see `scoring.py` docstring for full
rationale):**

| Gap | Decision made | Alternative to discuss |
|---|---|---|
| Missing protein data (74% of candidates) | Redistribute its weight onto RNA + tissue per-row | Could instead impute a value, or just accept the penalty |
| "Data consistency" definition | Percentile-rank agreement between RNA and protein | Could use raw correlation, or a fixed-bin agreement score |
| "Tissue relevance" target | Auto-detected enrichment (no target needed) *or* explicit `target_lineage` | Recommend exposing `target_lineage` as a user input |
| Exclusion: hard filter vs. soft penalty | Hard filter (score = -1, sorted to bottom, kept for audit) | Soft penalty risks excluded lines still ranking high |

**Suggested next step:** test this against 2-3 more genes with different
RNA/protein coverage profiles (e.g. a gene barely covered in the
proteomics file) to see if the missing-data handling holds up, then bring
the `target_lineage` design question to the group for a decision.